# Prototyping: Fraud Velocity Scoring

Prototype the 1-hour sliding window velocity aggregation for fraud scoring.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, window, count, sum as _sum, avg as _avg, max as _max,
    when, lit, expr, countDistinct
)
from pyspark.sql.types import IntegerType, DoubleType

spark = SparkSession.builder.getOrCreate()
df = spark.read.csv("data/sample_paysim.csv", header=True, inferSchema=True)

from pyspark.sql.functions import current_timestamp, to_timestamp
df = df.withColumn("event_time", to_timestamp(col("step"), "ss"))
df.printSchema()

In [ ]:
HIGH_VALUE_THRESHOLD = 10000.0

velocity = (
    df.groupBy(
        col("nameOrig"),
        window(col("event_time"), "1 hour", "15 minutes"),
    )
    .agg(
        count("*").alias("tx_count"),
        _sum("amount").alias("total_amount"),
        _avg("amount").alias("avg_amount"),
        _max("amount").alias("max_amount"),
        _sum(when(col("isFraud") == 1, 1).otherwise(0)).alias("fraud_count"),
        _sum(when(col("amount") >= HIGH_VALUE_THRESHOLD, 1).otherwise(0)).alias("high_value_count"),
    )
    .withColumn("window_start", col("window.start"))
    .withColumn("window_end", col("window.end"))
    .drop("window")
)

print("Velocity Metrics (sample):")
velocity.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import min as _min, max as _max

scored = velocity.withColumn(
    "fraud_score",
    when((col("high_value_count") >= 3) | (col("fraud_count") > 0), 0.9)
    .when(col("high_value_count") >= 2, 0.7)
    .when(col("high_value_count") >= 1, 0.4)
    .otherwise(0.1),
).withColumn(
    "is_suspicious",
    when(col("fraud_score") >= 0.7, lit(True)).otherwise(lit(False)),
)

print("Suspicious Activity Summary:")
scored.groupBy("is_suspicious").count().show()
print("Score Distribution:")
scored.select("fraud_score").distinct().orderBy("fraud_score").show()